In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/ClassroomActivity"
!ls "$DATASET_PATH"


Arguing			Holding_Book	      Writing_On_Board
Eating_in_classroom	Holding_Mobile_Phone  Writting_on_Textbook
Explaining_the_Subject	Reading_Book
HandRaise		Sitting_on_Desk


In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/ClassroomActivity"

# Step 1 — list all subfolders (labels)
print("🎯 Labels found in dataset:\n")

for folder in sorted(os.listdir(DATASET_PATH)):
    full_path = os.path.join(DATASET_PATH, folder)

    # skip non-folder items (like .zip, .txt)
    if not os.path.isdir(full_path):
        continue

    # count only video files
    video_files = [f for f in os.listdir(full_path) if f.lower().endswith(('.mp4', '.avi', '.mov'))]

    print(f"📂 {folder:25} => {len(video_files)} videos")


🎯 Labels found in dataset:

📂 Arguing                   => 64 videos
📂 Eating_in_classroom       => 60 videos
📂 Explaining_the_Subject    => 186 videos
📂 HandRaise                 => 53 videos
📂 Holding_Book              => 85 videos
📂 Holding_Mobile_Phone      => 90 videos
📂 Reading_Book              => 91 videos
📂 Sitting_on_Desk           => 72 videos
📂 Writing_On_Board          => 118 videos
📂 Writting_on_Textbook      => 95 videos


In [ ]:
# === STEP 1: Import libraries ===
import os
import cv2
import numpy as np
import glob
import tensorflow as tf
from tensorflow.keras import layers, models

# ✅ Labels based on your dataset folders
classes = [
    "Arguing",
    "Eating_in_classroom",
    "Explaining_the_Subject",
    "HandRaise",
    "Holding_Book",
    "Holding_Mobile_Phone",
    "Reading_Book",
    "Sitting_on_Desk",
    "Writing_On_Board",
    "Writting_on_Textbook"
]

print("✅ Environment ready! Total selected labels:", len(classes))
for c in classes:
    print("📂", c)


✅ Environment ready! Total selected labels: 10
📂 Arguing
📂 Eating_in_classroom
📂 Explaining_the_Subject
📂 HandRaise
📂 Holding_Book
📂 Holding_Mobile_Phone
📂 Reading_Book
📂 Sitting_on_Desk
📂 Writing_On_Board
📂 Writting_on_Textbook


In [ ]:
import tensorflow_hub as hub
import tensorflow as tf

# Load pretrained I3D from TensorFlow Hub
I3D_URL = "https://tfhub.dev/deepmind/i3d-kinetics-400/1"

print("🔹 Loading pretrained I3D model...")
i3d_model = hub.KerasLayer(I3D_URL, trainable=False)
print("✅ I3D model loaded successfully!")


🔹 Loading pretrained I3D model...
✅ I3D model loaded successfully!


In [ ]:
# ===== FULL FIXED I3D TEMPORAL FEATURE EXTRACTION PIPELINE =====
import os, glob
import numpy as np
import cv2
import tensorflow as tf
import tensorflow_hub as hub
from tqdm import tqdm

# optional fast reader
try:
    import imageio.v3 as iio
    IMAGEIO_AVAILABLE = True
except Exception:
    IMAGEIO_AVAILABLE = False

# ---- CONFIG ----
DATASET_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/ClassroomActivity"
FEATURE_SAVE_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/features_fixed"
os.makedirs(FEATURE_SAVE_PATH, exist_ok=True)

TARGET_SIZE = (224, 224)
MAX_FRAMES = 96
WINDOW_SIZE = 16
TIMESTEPS = MAX_FRAMES // WINDOW_SIZE
SAVE_EVERY = 20
VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv')

# ---- WORKING I3D MODEL ----
I3D_URL = "https://tfhub.dev/deepmind/i3d-kinetics-400/1"
print("Loading working I3D model...")
i3d_model = hub.KerasLayer(I3D_URL, trainable=False)

# ---- ROBUST FEATURE EXTRACTOR ----
def extract_feature_safe(window_batch_5d):
    """
    Extract feature for a window: handles different TF-Hub return types.
    window_batch_5d: numpy or tf.Tensor (1, T, H, W, C)
    Returns: 1D numpy array (feat_dim,)
    """
    if not isinstance(window_batch_5d, tf.Tensor):
        window_batch_5d = tf.convert_to_tensor(window_batch_5d, dtype=tf.float32)

    outputs = i3d_model(window_batch_5d)

    # Case A: dict-like
    if isinstance(outputs, dict) or hasattr(outputs, 'get'):
        for key in ("global_pool", "pooled", "feature", "embedding", "default"):
            if key in outputs:
                return tf.squeeze(outputs[key], axis=0).numpy()
        try:
            first_val = next(iter(outputs.values()))
            return tf.squeeze(first_val, axis=0).numpy()
        except Exception:
            pass

    # Case B: tuple/list
    if isinstance(outputs, (list, tuple)):
        for maybe in outputs:
            if isinstance(maybe, (tf.Tensor, np.ndarray)):
                return tf.squeeze(maybe).numpy()
        try:
            return tf.squeeze(tf.convert_to_tensor(outputs)).numpy()
        except Exception:
            pass

    # Case C: single Tensor
    if isinstance(outputs, tf.Tensor):
        return tf.squeeze(outputs).numpy()

    # Fallback
    try:
        return np.squeeze(np.array(outputs))
    except Exception as e:
        raise RuntimeError("Unable to extract feature from I3D output: " + str(e))

# ---- HELPER FUNCTIONS ----
def read_all_frames(video_path, target_size=TARGET_SIZE):
    frames = []
    if IMAGEIO_AVAILABLE:
        try:
            for f in iio.imiter(video_path, plugin="pyav"):
                if f is None:
                    continue
                f = cv2.resize(f, target_size)
                f = f.astype('float32') / 255.0
                frames.append(f)
        except Exception:
            frames = []

    if not frames:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            try:
                frame = cv2.resize(frame, target_size)
            except Exception:
                continue
            frame = frame.astype('float32') / 255.0
            frames.append(frame)
        cap.release()
    return frames

def uniform_sample_frames(frames, target_count=MAX_FRAMES):
    n = len(frames)
    if n == 0:
        return None
    if n >= target_count:
        idx = np.linspace(0, n - 1, target_count).astype(int)
        return np.array([frames[i] for i in idx], dtype=np.float32)
    sampled = frames.copy()
    while len(sampled) < target_count:
        sampled.append(sampled[-1])
    return np.array(sampled[:target_count], dtype=np.float32)

def frames_to_windows(sampled_frames, window_size=WINDOW_SIZE):
    windows = []
    for t in range(TIMESTEPS):
        start = t * window_size
        end = start + window_size
        windows.append(sampled_frames[start:end])
    return np.stack(windows, axis=0)  # (TIMESTEPS, WINDOW_SIZE, H, W, C)

# ---- MAIN EXTRACTION LOOP ----
parts = 0
X_batch, y_batch = [], []
total = 0
FEAT_DIM = None

labels = [d for d in sorted(os.listdir(DATASET_PATH)) if os.path.isdir(os.path.join(DATASET_PATH, d))]
print("Found labels:", labels)

for label in labels:
    folder = os.path.join(DATASET_PATH, label)
    videos = [f for f in sorted(os.listdir(folder)) if f.lower().endswith(VIDEO_EXTS)]
    print(f"\nProcessing label: {label} ({len(videos)} videos)")

    for v in tqdm(videos):
        video_path = os.path.join(folder, v)
        try:
            frames = read_all_frames(video_path)
            if not frames:
                print("Skipped:", video_path)
                continue

            sampled = uniform_sample_frames(frames)
            if sampled is None:
                print("Skipped:", video_path)
                continue

            windows = frames_to_windows(sampled)

            feat_list = []
            for w in windows:
                w_batch = np.expand_dims(w, axis=0)
                try:
                    feat_vec = extract_feature_safe(w_batch)
                    # record feature dimension on first run
                    if FEAT_DIM is None:
                        FEAT_DIM = feat_vec.shape[0]
                except Exception as e:
                    print("Feature extraction failed, using zeros:", e)
                    if FEAT_DIM is not None:
                        feat_vec = np.zeros((FEAT_DIM,), dtype=np.float32)
                    else:
                        raise e
                feat_list.append(feat_vec)

            feat_seq = np.stack(feat_list, axis=0)  # (TIMESTEPS, FEAT_DIM)

            X_batch.append(feat_seq)
            y_batch.append(label)
            total += 1

            # Save periodically
            if total % SAVE_EVERY == 0:
                np.save(os.path.join(FEATURE_SAVE_PATH, f"X_part_{parts}.npy"), np.array(X_batch))
                np.save(os.path.join(FEATURE_SAVE_PATH, f"y_part_{parts}.npy"), np.array(y_batch))
                print(f"Saved part {parts}: X_batch shape {np.array(X_batch).shape}")
                parts += 1
                X_batch, y_batch = [], []

        except Exception as e:
            print("Error:", video_path, "->", e)
            continue

# leftover save
if X_batch:
    np.save(os.path.join(FEATURE_SAVE_PATH, f"X_part_{parts}.npy"), np.array(X_batch))
    np.save(os.path.join(FEATURE_SAVE_PATH, f"y_part_{parts}.npy"), np.array(y_batch))
    print(f"Saved final part {parts}")
    parts += 1

print("Extraction done. Total processed:", total)

# ---- MERGE ALL PARTS ----
parts_x = sorted(glob.glob(os.path.join(FEATURE_SAVE_PATH, "X_part_*.npy")))
parts_y = sorted(glob.glob(os.path.join(FEATURE_SAVE_PATH, "y_part_*.npy")))

if parts_x:
    X_all = np.concatenate([np.load(p) for p in parts_x], axis=0)
    y_all = np.concatenate([np.load(p) for p in parts_y], axis=0)
    np.save(os.path.join(FEATURE_SAVE_PATH, "X_features.npy"), X_all)
    np.save(os.path.join(FEATURE_SAVE_PATH, "y_labels.npy"), y_all)
    print("Final features saved.")
    print("X shape:", X_all.shape, "y shape:", y_all.shape)
else:
    print("No parts found to merge!")


Loading working I3D model...
Found labels: ['Arguing', 'Eating_in_classroom', 'Explaining_the_Subject', 'HandRaise', 'Holding_Book', 'Holding_Mobile_Phone', 'Reading_Book', 'Sitting_on_Desk', 'Writing_On_Board', 'Writting_on_Textbook']

Processing label: Arguing (64 videos)


 31%|███▏      | 20/64 [04:20<13:47, 18.80s/it]

Saved part 0: X_batch shape (20, 6, 400)


 62%|██████▎   | 40/64 [11:21<09:05, 22.74s/it]

Saved part 1: X_batch shape (20, 6, 400)


...

Saved part 43: X_batch shape (20, 6, 400)


 85%|████████▌ | 81/95 [18:36<04:07, 17.65s/it]

Saved part 44: X_batch shape (20, 6, 400)


100%|██████████| 95/95 [22:55<00:00, 14.47s/it]


Saved final part 45
Extraction done. Total processed: 914
Final features saved.
X shape: (914, 6, 400) y shape: (914,)


In [ ]:
import os
FEATURE_SAVE_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/features_fixed"

# List files in the folder to verify
print(os.listdir(FEATURE_SAVE_PATH))

# Load
X = np.load(os.path.join(FEATURE_SAVE_PATH, "X_features.npy"))
y = np.load(os.path.join(FEATURE_SAVE_PATH, "y_labels.npy"))


['X_part_0.npy', 'y_part_0.npy', 'X_part_1.npy', 'y_part_1.npy', 'X_part_2.npy', 'y_part_2.npy', 'X_part_3.npy', 'y_part_3.npy', 'X_part_4.npy', 'y_part_4.npy', 'X_part_5.npy', 'y_part_5.npy', 'X_part_6.npy', 'y_part_6.npy', 'X_part_7.npy', 'y_part_7.npy', 'X_part_8.npy', 'y_part_8.npy', 'X_part_9.npy', 'y_part_9.npy', 'X_part_10.npy', 'y_part_10.npy', 'X_part_11.npy', 'y_part_11.npy', 'X_part_12.npy', 'y_part_12.npy', 'X_part_13.npy', 'y_part_13.npy', 'X_part_14.npy', 'y_part_14.npy', 'X_part_15.npy', 'y_part_15.npy', 'X_part_16.npy', 'y_part_16.npy', 'X_part_17.npy', 'y_part_17.npy', 'X_part_18.npy', 'y_part_18.npy', 'X_part_19.npy', 'y_part_19.npy', 'X_part_20.npy', 'y_part_20.npy', 'X_part_21.npy', 'y_part_21.npy', 'X_part_22.npy', 'y_part_22.npy', 'X_part_23.npy', 'y_part_23.npy', 'X_part_24.npy', 'y_part_24.npy', 'X_part_25.npy', 'y_part_25.npy', 'X_part_26.npy', 'y_part_26.npy', 'X_part_27.npy', 'y_part_27.npy', 'X_part_28.npy', 'y_part_28.npy', 'X_part_29.npy', 'y_part_29.npy',

In [ ]:
# === STEP 1: Imports ===
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# === STEP 2: Load features and labels ===
FEATURE_SAVE_PATH = "/content/drive/MyDrive/dataset/ClassroomActivity/features_fixed"

X = np.load(FEATURE_SAVE_PATH + "/X_features.npy")  # shape: (914, 6, 400)
y = np.load(FEATURE_SAVE_PATH + "/y_labels.npy")    # shape: (914,)
print(X.shape, y.shape)  # should print (914, 6, 400)

# === STEP 3: Encode labels ===
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
print("✅ Classes:", le.classes_)

# Convert to categorical
y_cat = tf.keras.utils.to_categorical(y_encoded, num_classes=num_classes)

# === STEP 4: Train-test split ===
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y_encoded
)
print("✅ Training samples:", X_train.shape[0], "Validation samples:", X_val.shape[0])

# === STEP 5: Reshape for LSTM ===
# LSTM expects 3D input: (samples, timesteps, features)
# Our X is already (samples, 6, 400) => no need to reshape
X_train_lstm = X_train
X_val_lstm = X_val
print("✅ LSTM input shape:", X_train_lstm.shape)

# === STEP 6: Build LSTM model ===
model = models.Sequential()
model.add(layers.LSTM(256, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=False))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(num_classes, activation='softmax'))

# === STEP 7: Compile ===
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# === STEP 8: Callbacks ===
checkpoint_cb = callbacks.ModelCheckpoint(FEATURE_SAVE_PATH + "/lstm_model.h5", save_best_only=True)
earlystop_cb = callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# === STEP 9: Train ===
history = model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=30,
    batch_size=16,
    callbacks=[checkpoint_cb, earlystop_cb]
)

# === STEP 10: Save final model ===
model.save(FEATURE_SAVE_PATH + "/fulfinal_model.h5")
print("✅ Model training complete and saved!")


(914, 6, 400) (914,)
✅ Classes: ['Arguing' 'Eating_in_classroom' 'Explaining_the_Subject' 'HandRaise'
 'Holding_Book' 'Holding_Mobile_Phone' 'Reading_Book' 'Sitting_on_Desk'
 'Writing_On_Board' 'Writting_on_Textbook']
✅ Training samples: 731 Validation samples: 183
✅ LSTM input shape: (731, 6, 400)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 256)            │       672,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 706,954 (2.70 MB)

...

46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.8995 - loss: 0.3168 - val_accuracy: 0.8852 - val_loss: 0.3286
Epoch 13/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.8820 - loss: 0.3232 - val_accuracy: 0.8852 - val_loss: 0.3468
Epoch 14/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.8961 - loss: 0.3165 - val_accuracy: 0.8907 - val_loss: 0.3332
Epoch 15/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8781 - loss: 0.3488 - val_accuracy: 0.8907 - val_loss: 0.3709
Epoch 16/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9166 - loss: 0.2548 - val_accuracy: 0.8852 - val_loss: 0.3679
Epoch 17/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9124 - loss: 0.2301 - val_accuracy: 0.9071 - val_loss: 0.3297
Epoch 18/30
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9366 - loss: 0.1951

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9365 - loss: 0.1958 - val_accuracy: 0.9016 - val_loss: 0.3216
Epoch 19/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8887 - loss: 0.2830 - val_accuracy: 0.8962 - val_loss: 0.3485
Epoch 20/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9239 - loss: 0.2217 - val_accuracy: 0.8689 - val_loss: 0.3519
Epoch 21/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9118 - loss: 0.2359 - val_accuracy: 0.8798 - val_loss: 0.3299
Epoch 22/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9300 - loss: 0.2032 - val_accuracy: 0.8962 - val_loss: 0.3585
Epoch 23/30
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9305 - loss: 0.1873

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9303 - loss: 0.1882 - val_accuracy: 0.9016 - val_loss: 0.3213
Epoch 24/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9369 - loss: 0.1752

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9368 - loss: 0.1754 - val_accuracy: 0.9180 - val_loss: 0.3160
Epoch 25/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9305 - loss: 0.1907 - val_accuracy: 0.8962 - val_loss: 0.3465
Epoch 26/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9569 - loss: 0.1456 - val_accuracy: 0.8962 - val_loss: 0.3420
Epoch 27/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9524 - loss: 0.1451 - val_accuracy: 0.8907 - val_loss: 0.3435
Epoch 28/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9418 - loss: 0.1585 - val_accuracy: 0.8852 - val_loss: 0.3583
Epoch 29/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9334 - loss: 0.1836 - val_accuracy: 0.8962 - val_loss: 0.3496
Epoch 30/30
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9564 - loss: 0.1665 - val_accuracy: 0.8962 - val_loss: 0.3476


✅ Model training complete and saved!


In [ ]:
# ============================
# 🎯 STEP 1 — IMPORTS
# ============================
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from collections import Counter
import os

# ============================
# 🎯 STEP 2 — CONFIG
# ============================
VIDEO_PATH = "/content/drive/MyDrive/c447.mp4"   # 🔴 CHANGE if needed
SAVE_PATH  = "/content/drive/MyDrive/video 00.mp4"

TARGET_SIZE = (224, 224)
MAX_FRAMES = 96
WINDOW_SIZE = 16
TIMESTEPS = MAX_FRAMES // WINDOW_SIZE
CONF_THRESHOLD = 0.5   # 🔥 slightly lower for testing

class_labels = [
    "Arguing","Eating_in_classroom","Explaining_the_Subject",
    "HandRaise","Holding_Book","Holding_Mobile_Phone",
    "Reading_Book","Sitting_on_Desk","Writing_On_Board",
    "Writting_on_Textbook"
]

# ============================
# 🎯 STEP 3 — CHECK VIDEO FILE
# ============================

print("🔍 Checking video path...")

if not os.path.exists(VIDEO_PATH):
    print("❌ Video file NOT found at:", VIDEO_PATH)
    print("👉 Upload video or fix path!")
    exit()

print("✅ Video file found!")

# ============================
# 🎯 STEP 4 — LOAD MODELS
# ============================
print("⏳ Loading models...")

i3d_model = hub.KerasLayer(
    "https://tfhub.dev/deepmind/i3d-kinetics-400/1",
    trainable=False
)

lstm_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/dataset/ClassroomActivity/features_fixed/full_final_model.h5"
)

print("✅ Models loaded!")

# ============================
# 🎯 STEP 5 — HELPER FUNCTIONS
# ============================

def read_frames(cap, start_frame, num_frames):
    frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    for _ in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, TARGET_SIZE)
        frame = frame.astype("float32") / 255.0
        frames.append(frame)

    if len(frames) == 0:
        return None

    # repeat last frame
    while len(frames) < MAX_FRAMES:
        frames.append(frames[-1])

    return np.array(frames[:MAX_FRAMES], dtype=np.float32)


def to_windows(frames):
    return np.array([
        frames[i*WINDOW_SIZE:(i+1)*WINDOW_SIZE]
        for i in range(TIMESTEPS)
    ])


def extract_i3d_vec(window):
    window = np.expand_dims(window, axis=0)
    feat = i3d_model(window)
    return tf.squeeze(feat).numpy()


def to_lstm_feature(windows):
    feats = [extract_i3d_vec(w) for w in windows]
    return np.array(feats).reshape(1, TIMESTEPS, -1)

# ============================
# 🎯 STEP 6 — PROCESS VIDEO
# ============================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("❌ Cannot open video")
    exit()

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS = int(cap.get(cv2.CAP_PROP_FPS))

if FPS == 0:
    print("⚠️ FPS issue → using default 30")
    FPS = 30

print(f"🎥 Total Frames: {total_frames}, FPS: {FPS}")

chunk_size = 4 * FPS
pred_segments = []

print("\n⏳ Processing video...")

for start in range(0, total_frames, chunk_size):

    frames = read_frames(cap, start, chunk_size)
    if frames is None:
        continue

    windows = to_windows(frames)
    lstm_input = to_lstm_feature(windows)

    preds = lstm_model.predict(lstm_input, verbose=0)
    label = class_labels[np.argmax(preds)]
    conf  = float(np.max(preds))

    print(f"Segment {start/FPS:.1f}s → {label} ({conf:.2f})")

    if conf >= CONF_THRESHOLD:
        pred_segments.append([label, conf, start, start + chunk_size])

cap.release()

# ============================
# 🎯 STEP 7 — HANDLE EMPTY CASE
# ============================

if len(pred_segments) == 0:
    print("\n❌ No confident segments found!")
    print("👉 Try lowering threshold OR change video")
    exit()

# ============================
# 🎯 STEP 8 — MERGE SEGMENTS
# ============================

merged = []

for seg in pred_segments:
    if not merged:
        merged.append(seg)
    else:
        last = merged[-1]
        if last[0] == seg[0]:
            last[3] = seg[3]
        else:
            merged.append(seg)

pred_segments = merged

# ============================
# 🎯 STEP 9 — TIMELINE SUMMARY
# ============================

print("\n📊 Timeline Summary:\n")

for label, conf, s, e in pred_segments:
    print(f"{s/FPS:.1f}s - {e/FPS:.1f}s → {label} ({conf*100:.1f}%)")

# ============================
# 🎯 STEP 10 — NLP SUMMARY
# ============================

labels_only = [p[0] for p in pred_segments]
top = Counter(labels_only).most_common(2)

summary = "The video mostly shows "

if len(top) == 1:
    summary += top[0][0]
else:
    summary += f"{top[0][0]} and {top[1][0]}"

summary += " activities."

print("\n🧠 AI Summary:")
print(summary)

# ============================
# 🎯 STEP 11 — CREATE VIDEO
# ============================

cap = cv2.VideoCapture(VIDEO_PATH)

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(SAVE_PATH,
                      cv2.VideoWriter_fourcc(*"mp4v"),
                      FPS, (w, h))

print("\n🎬 Creating summary video...")

for label, conf, s, e in pred_segments:
    cap.set(cv2.CAP_PROP_POS_FRAMES, s)

    for _ in range(s, e):
        ret, frame = cap.read()
        if not ret:
            break

        text = f"{label} ({conf*100:.1f}%)"

        cv2.rectangle(frame, (20, 20), (650, 100), (0,0,0), -1)
        cv2.putText(frame, text, (30, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5, (0,0,255), 3)

        out.write(frame)

cap.release()
out.release()

print("\n🎉 DONE! Video saved at:", SAVE_PATH)

SyntaxError: expected ':' (3967160605.py, line 208)

In [ ]:
import numpy as np
from collections import defaultdict

# === Compute overall model confidence ===
all_confidences = [conf for _, conf in predictions if conf is not None]

if len(all_confidences) > 0:
    overall_confidence = np.mean(all_confidences)
else:
    overall_confidence = 0.0

print(f"\n📊 Overall Model Confidence: {overall_confidence * 100:.2f}%")

# === Generate Descriptive Summary (3–4 lines) ===
label_conf = defaultdict(list)
for item in predictions:
    if len(item) >= 2:
        label, conf = item[0], item[1]
        label_conf[label].append(conf)

# Average confidence per label
avg_conf = {lbl: np.mean(vals) for lbl, vals in label_conf.items()}
# Sort labels by descending confidence
sorted_labels = sorted(avg_conf.items(), key=lambda x: x[1], reverse=True)
top_labels = [lbl for lbl, _ in sorted_labels[:4]]

# Map model labels to readable descriptions
READABLE = {
    "Arguing": "a teacher or student having a discussion or argument in class",
    "Explaining_the_Subject": "the teacher explaining concepts on the board",
    "HandRaise": "students raising their hands to answer or ask questions",
    "Holding_Book": "the teacher holding and referring to a textbook",
    "Holding_Mobile_Phone": "students or teacher using a mobile device",
    "Reading_Book": "students reading their textbooks attentively",
    "Sitting_on_Desk": "students sitting on their desks",
    "Writing_On_Board": "the teacher writing important points on the board",
    "Writting_on_Textbook": "students writing notes in their textbooks",
    "Eating_in_classroom": "students eating or snacking in class"
}

summary_parts = [READABLE.get(lbl, lbl) for lbl in top_labels]

video_summary = "This classroom video primarily captures various teaching and learning moments. "

if len(summary_parts) == 1:
    video_summary += f"It mainly focuses on {summary_parts[0]}. "
elif len(summary_parts) == 2:
    video_summary += (
        f"The video mostly shows {summary_parts[0]}, followed by scenes of {summary_parts[1]}. "
    )
elif len(summary_parts) == 3:
    video_summary += (
        f"The video first shows {summary_parts[0]}, then {summary_parts[1]}, "
        f"and later {summary_parts[2]}. "
    )
else:
    video_summary += (
        f"The video begins with {summary_parts[0]}, transitions to {summary_parts[1]}, "
        f"continues with {summary_parts[2]}, and concludes showing {summary_parts[3]}. "
    )

video_summary += (
    f"Overall, the model analyzed the video with an average confidence of "
    f"{overall_confidence * 100:.1f}%, indicating a fairly reliable classification performance."
)

print("\n📝 Detailed Video Summary:\n")
print(video_summary)



📊 Overall Model Confidence: 72.44%

📝 Detailed Video Summary:

This classroom video primarily captures various teaching and learning moments. The video begins with a teacher or student having a discussion or argument in class, transitions to students writing notes in their textbooks, continues with the teacher explaining concepts on the board, and concludes showing students or teacher using a mobile device. Overall, the model analyzed the video with an average confidence of 72.4%, indicating a fairly reliable classification performance.


In [ ]:
import cv2
import numpy as np

# ============================
# 🎯 CONFIG
# ============================

VIDEO_PATH = "/content/5427175-hd_1920_1080_25fps.mp4"
SAVE_PATH  = "/content/output 0.mp4"

# ============================
# 🎯 LOAD VIDEO
# ============================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("❌ Video not found!")
    exit()

FPS = int(cap.get(cv2.CAP_PROP_FPS))
if FPS == 0:
    FPS = 30

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# ============================
# 🎯 VIDEO WRITER
# ============================

out = cv2.VideoWriter(
    SAVE_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (frame_w, frame_h)
)

# ============================
# 🎯 PERSON DETECTOR
# ============================

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# ============================
# 🎯 PROCESS VIDEO
# ============================

teacher_frames = []
prev_gray = None
frame_idx = 0

print("⏳ Processing video...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---- Detect people ----
    boxes, _ = hog.detectMultiScale(frame, winStride=(8,8))

    # ---- Motion detection ----
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    motion_map = None

    if prev_gray is not None:
        motion_map = cv2.absdiff(prev_gray, gray)

    prev_gray = gray

    # ============================
    # 🎯 SELECT ONLY ONE TEACHER
    # ============================

    best_score = -1
    teacher_box = None
    valid_boxes = []

    for (x, y, w, h) in boxes:

        # ❗ Ignore small detections
        if w * h < 4000:
            continue

        valid_boxes.append((x, y, w, h))

        score = 0

        # 1️⃣ Center preference
        center_x = x + w // 2
        if frame_w * 0.3 < center_x < frame_w * 0.7:
            score += 2

        # 2️⃣ Bottom preference (teacher near camera)
        if y > frame_h * 0.4:
            score += 2

        # 3️⃣ Size (larger = closer)
        score += (w * h) / 8000

        # 4️⃣ Motion
        if motion_map is not None:
            person_motion = motion_map[y:y+h, x:x+w]
            if np.sum(person_motion) > 30000:
                score += 1

        # Select best candidate
        if score > best_score:
            best_score = score
            teacher_box = (x, y, w, h)

    # ============================
    # 🎯 DRAW RESULTS
    # ============================

    for (x, y, w, h) in valid_boxes:

        if teacher_box is not None and (x, y, w, h) == teacher_box:
            label = "Teacher"
            color = (0,255,0)
            teacher_frames.append(frame_idx)
        else:
            label = "Student"
            color = (255,0,0)

        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
        cv2.putText(frame, label, (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, color, 2)

    # ---- Save frame ----
    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

print("✅ Video saved at:", SAVE_PATH)

# ============================
# 🎯 PUNCTUALITY
# ============================

print("\n⏰ Punctuality Result:\n")

if len(teacher_frames) > 0:

    arrival_time = min(teacher_frames) / FPS
    departure_time = max(teacher_frames) / FPS

    print(f"📌 Arrival Time: {arrival_time:.2f} sec")
    print(f"📌 Departure Time: {departure_time:.2f} sec")

    CLASS_START = 0
    CLASS_END = total_frames / FPS
    GRACE = 10

    if arrival_time <= CLASS_START + GRACE:
        arrival_status = "On-Time"
    else:
        arrival_status = "Late"

    if departure_time >= CLASS_END - GRACE:
        departure_status = "Full Duration"
    else:
        departure_status = "Early Leave"

    if arrival_status == "On-Time" and departure_status == "Full Duration":
        final_status = "ON-TIME"
    elif arrival_status == "Late":
        final_status = "LATE ARRIVAL"
    elif departure_status == "Early Leave":
        final_status = "EARLY LEAVE"
    else:
        final_status = "IRREGULAR"

    print("\n📊 Final Status:", final_status)

else:
    print("❌ Teacher not detected → ABSENT")

⏳ Processing video...
✅ Video saved at: /content/output 0.mp4

⏰ Punctuality Result:

📌 Arrival Time: 0.00 sec
📌 Departure Time: 19.44 sec

📊 Final Status: ON-TIME
